# 03 - 模型 API 服务

## 学习目标
- 将训练好的模型封装为 API
- 实现预测接口
- Docker 容器化部署

In [ ]:
import joblib
import numpy as np
from pathlib import Path

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
import json

## 1. 训练并保存模型

In [ ]:
# 加载数据
data = fetch_california_housing()
X = data.data
y = data.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 标准化
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 训练模型
model = LinearRegression()
model.fit(X_train_scaled, y_train)

print(f'训练 R²: {model.score(X_test_scaled, y_test):.4f}')

# 保存模型
save_dir = Path('../../models_saved/sklearn')
save_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(model, save_dir / 'housing_model.pkl')
joblib.dump(scaler, save_dir / 'housing_scaler.pkl')

print(f'✓ 模型已保存')

## 2. 完整 API 代码

In [ ]:
api_code = '''
"""
model_api.py - 房价预测 API
"""

from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
from typing import List
import joblib
import numpy as np
from pathlib import Path

# 初始化
app = FastAPI(
    title="房价预测 API",
    description="基于加州房价数据集的线性回归模型",
    version="1.0.0"
)

# 加载模型
MODEL_DIR = Path("../../models_saved/sklearn")
model = joblib.load(MODEL_DIR / "housing_model.pkl")
scaler = joblib.load(MODEL_DIR / "housing_scaler.pkl")

# 数据模型
class HouseFeatures(BaseModel):
    MedInc: float = Field(..., description="收入中位数")
    HouseAge: float = Field(..., description="房屋年龄")
    AveRooms: float = Field(..., description="平均房间数")
    AveBedrms: float = Field(..., description="平均卧室数")
    Population: float = Field(..., description="人口")
    AveOccup: float = Field(..., description="平均入住率")
    Latitude: float = Field(..., description="纬度")
    Longitude: float = Field(..., description="经度")

class PredictionResponse(BaseModel):
    predicted_price: float
    features: dict

# 路由
@app.get("/")
async def root():
    return {"message": "房价预测 API", "version": "1.0.0"}

@app.get("/health")
async def health():
    return {"status": "healthy", "model_loaded": True}

@app.post("/predict", response_model=PredictionResponse)
async def predict(house: HouseFeatures):
    try:
        features = np.array([[
            house.MedInc, house.HouseAge, house.AveRooms,
            house.AveBedrms, house.Population, house.AveOccup,
            house.Latitude, house.Longitude
        ]])
        
        features_scaled = scaler.transform(features)
        prediction = model.predict(features_scaled)[0]
        
        return PredictionResponse(
            predicted_price=round(float(prediction), 4),
            features=house.dict()
        )
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

@app.post("/predict/batch")
async def predict_batch(houses: List[HouseFeatures]):
    features = np.array([
        [h.MedInc, h.HouseAge, h.AveRooms, h.AveBedrms,
         h.Population, h.AveOccup, h.Latitude, h.Longitude]
        for h in houses
    ])
    
    features_scaled = scaler.transform(features)
    predictions = model.predict(features_scaled)
    
    return {
        "predictions": [round(float(p), 4) for p in predictions],
        "count": len(predictions)
    }
'''

print(api_code)

## 3. Docker 部署

In [ ]:
dockerfile = '''
FROM python:3.10-slim

WORKDIR /app

# 安装依赖
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# 复制代码
COPY . .

# 暴露端口
EXPOSE 8000

# 启动命令
CMD ["uvicorn", "model_api:app", "--host", "0.0.0.0", "--port", "8000"]
'''

print("Dockerfile:")
print(dockerfile)

In [ ]:
docker_commands = '''
# 构建镜像
docker build -t ml-api .

# 运行容器
docker run -p 8000:8000 ml-api

# 后台运行
docker run -d -p 8000:8000 --name ml-api ml-api

# 查看日志
docker logs ml-api

# 停止容器
docker stop ml-api
'''

print("Docker 命令:")
print(docker_commands)

## 4. 测试 API

In [ ]:
# 模拟 API 调用
test_data = {
    "MedInc": 8.3252,
    "HouseAge": 41.0,
    "AveRooms": 6.984,
    "AveBedrms": 1.023,
    "Population": 322.0,
    "AveOccup": 2.555,
    "Latitude": 37.88,
    "Longitude": -122.23
}

# 本地测试（不使用 API）
features = np.array([[list(test_data.values())]])
features_scaled = scaler.transform(features)
prediction = model.predict(features_scaled)[0]

print(f'输入特征: {json.dumps(test_data, indent=2)}')
print(f'\n预测房价: ${prediction * 100000:.0f}')

## 小结
- FastAPI 提供高性能的 API 服务
- Docker 确保环境一致性
- 模型服务化是 ML 项目的重要环节

---
🎉 恭喜！你已经完成了整个学习路径！